# 29 · Datasets et préparation du projet final — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch29_datasets/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`. Tout tourne sur CPU en quelques minutes. 29.10 et 29.11 interrogent le Hugging Face Hub et OpenML : sans réseau, leurs vérifications passent quand même (copie locale, services simulés) et les parties en direct sont sautées.

> Travaille dans **ta copie** (`mon_travail/ch29_datasets/03_notebook.ipynb`, créée par `python tools/start_chapter.py 29`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 29.Q1 à 29.Q8, 29.R1 à 29.R3, 29.1, 29.2 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ | |
| A | 29.7 | Les datasets intégrés de scikit-learn, et une affirmation du livre à vérifier | 📦 | ★ | 15 |
|  | 29.8 | Prédire : le même modèle sur MNIST puis sur Fashion-MNIST | 🔮 | ★ | 10 |
|  | 29.9 | Prédire la chute d'accuracy en changeant de source de chiffres | 🔮 | ★ | 10 |
|  | 29.10 | torchvision.datasets et Hugging Face datasets : mêmes données, deux API | 📦 | ★★ | 25 |
|  | 29.11 | Explorer le Hugging Face Hub et OpenML : chercher, filtrer, lire la licence | 📦 | ★★ | 25 |
| B | 29.12 | Profil express d'un dataset inconnu | 📈 | ★★ | 20 |
|  | 29.13 | Un score trop beau pour être vrai : doublons entre entraînement et test | 🐛 | ★★ | 25 |
|  | 29.14 | Décalage de distribution : un modèle MNIST face aux chiffres de scikit-learn | 🔬 | ★★★ | 35 |
|  | 29.15 | Data card et script de téléchargement vérifié pour le projet final | 🛠️ | ★★★ | 60 |
|  | 29.16 | Le dataset piégé : trouver les huit défauts | 🏆 | ★★★ | 60 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="29")  # your own library

## Objectifs et rappel express

- Charger des datasets avec scikit-learn, torchvision et Hugging Face `datasets` (à une révision fixée), et chercher sur le Hub et OpenML en lisant la licence.
- Profiler un tableau inconnu, détecter les doublons qui fuient entre entraînement et test, et découper par groupes.
- Mesurer un décalage de distribution et ce que récupèrent une meilleure préparation et quelques exemples étiquetés.
- Écrire un téléchargement vérifié par SHA-256 et la data card du dataset de ton projet final.

**Rappel express.** Mesurer sur les données chargées, pas croire la description. Un dataset public n'est pas forcément libre : lire sa licence. Découper **avant** toute transformation ajustée sur les données, et **par groupes** quand des exemples sont liés. Mémoire d'un dataset d'images : $N \times H \times W \times C \times$ (octets par valeur).

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 29.1 et 29.2)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; plusieurs lettres : entre guillemets (`"ABD"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », l'estimation 🧮 29.3, la réflexion (🗣️ 29.4, ⚖️ 29.5, 📄 29.6) et l'entretien se corrigent avec `05_solutions.md`.

**29.Q1 — Partir de l'existant : modèle pré-entraîné, dataset public ou collecte ?**

In [ ]:
# 29.Q1: the values you found on paper
answer_29_Q1a = ...  # a) the letter of your choice, e.g. "B"
answer_29_Q1b = ...  # b) True or False
answer_29_Q1c = ...  # c) the letter of your choice, e.g. "B"
answer_29_Q1d = ...  # d) True or False
answer_29_Q1e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_29_Q1a, answer_29_Q1b, answer_29_Q1c, answer_29_Q1d, answer_29_Q1e]):
    wb.check(f"29.Q1{letter}", answer)

**29.Q2 — Licences et restrictions d'usage : vrai ou faux**

In [ ]:
# 29.Q2: the values you found on paper
answer_29_Q2a = ...  # a) True or False
answer_29_Q2b = ...  # b) True or False
answer_29_Q2c = ...  # c) True or False
answer_29_Q2d = ...  # d) True or False
answer_29_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_29_Q2a, answer_29_Q2b, answer_29_Q2c, answer_29_Q2d, answer_29_Q2e]):
    wb.check(f"29.Q2{letter}", answer)

**29.Q3 — Les défauts qu'une inspection manuelle révèle**

In [ ]:
# 29.Q3: the values you found on paper
answer_29_Q3a = ...  # a) the letters, e.g. "ABD"
answer_29_Q3b = ...  # b) the letter of your choice, e.g. "B"
answer_29_Q3c = ...  # c) True or False
answer_29_Q3d = ...  # d) the letter of your choice, e.g. "B"
answer_29_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_29_Q3a, answer_29_Q3b, answer_29_Q3c, answer_29_Q3d, answer_29_Q3e]):
    wb.check(f"29.Q3{letter}", answer)

**29.Q4 — MNIST et Fashion-MNIST : même format, difficulté différente**

In [ ]:
# 29.Q4: the values you found on paper
answer_29_Q4a = ...  # a) a whole number
answer_29_Q4b = ...  # b) a whole number
answer_29_Q4c = ...  # c) a whole number
answer_29_Q4d = ...  # d) True or False
answer_29_Q4e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_29_Q4a, answer_29_Q4b, answer_29_Q4c, answer_29_Q4d, answer_29_Q4e]):
    wb.check(f"29.Q4{letter}", answer)

**29.Q5 — Les datasets de scikit-learn : lequel pour quelle tâche, et pourquoi Boston a disparu**

In [ ]:
# 29.Q5: the values you found on paper
answer_29_Q5a = ...  # a) a whole number
answer_29_Q5b = ...  # b) the letter of your choice, e.g. "B"
answer_29_Q5c = ...  # c) the letter of your choice, e.g. "B"
answer_29_Q5d = ...  # d) the letter of your choice, e.g. "B"
answer_29_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_29_Q5a, answer_29_Q5b, answer_29_Q5c, answer_29_Q5d, answer_29_Q5e]):
    wb.check(f"29.Q5{letter}", answer)

**29.Q6 — Des datasets de Keras à torchvision et Hugging Face**

In [ ]:
# 29.Q6: the values you found on paper
answer_29_Q6a = ...  # a) the letter of your choice, e.g. "B"
answer_29_Q6b = ...  # b) a whole number
answer_29_Q6c = ...  # c) the letter of your choice, e.g. "B"
answer_29_Q6d = ...  # d) the letter of your choice, e.g. "B"
answer_29_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_29_Q6a, answer_29_Q6b, answer_29_Q6c, answer_29_Q6d, answer_29_Q6e]):
    wb.check(f"29.Q6{letter}", answer)

**29.Q7 — Collections de datasets : UCI, Kaggle, OpenML, Hugging Face Hub et liens morts**

In [ ]:
# 29.Q7: the values you found on paper
answer_29_Q7a = ...  # a) the letters, e.g. "ABD"
answer_29_Q7b = ...  # b) True or False
answer_29_Q7c = ...  # c) the letter of your choice, e.g. "B"
answer_29_Q7d = ...  # d) True or False
answer_29_Q7e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_29_Q7a, answer_29_Q7b, answer_29_Q7c, answer_29_Q7d, answer_29_Q7e]):
    wb.check(f"29.Q7{letter}", answer)

**29.Q8 — Les « nouveaux » datasets de 2018 : tâches, tailles et retraits**

In [ ]:
# 29.Q8: the values you found on paper
answer_29_Q8a = ...  # a) a whole number
answer_29_Q8b = ...  # b) the letters, e.g. "ABD"
answer_29_Q8c = ...  # c) the letter of your choice, e.g. "B"
answer_29_Q8d = ...  # d) the letter of your choice, e.g. "B"
answer_29_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_29_Q8a, answer_29_Q8b, answer_29_Q8c, answer_29_Q8d, answer_29_Q8e]):
    wb.check(f"29.Q8{letter}", answer)

**29.R1 — Qui possède les droits d'une image de style ?**

In [ ]:
# 29.R1: the values you found on paper
answer_29_R1a = ...  # a) a whole number
answer_29_R1b = ...  # b) True or False
answer_29_R1c = ...  # c) True or False
answer_29_R1d = ...  # d) the letter of your choice, e.g. "B"
answer_29_R1e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_29_R1a, answer_29_R1b, answer_29_R1c, answer_29_R1d, answer_29_R1e]):
    wb.check(f"29.R1{letter}", answer)

**29.R2 — En RL, d'où viennent les données ?**

In [ ]:
# 29.R2: the values you found on paper
answer_29_R2a = ...  # a) the letter of your choice, e.g. "B"
answer_29_R2b = ...  # b) True or False
answer_29_R2c = ...  # c) the letter of your choice, e.g. "B"
answer_29_R2d = ...  # d) a whole number
answer_29_R2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_29_R2a, answer_29_R2b, answer_29_R2c, answer_29_R2d, answer_29_R2e]):
    wb.check(f"29.R2{letter}", answer)

**29.R3 — Biais d'échantillonnage — un dataset représente-t-il la population visée ?**

In [ ]:
# 29.R3: the values you found on paper
answer_29_R3a = ...  # a) a whole number
answer_29_R3b = ...  # b) a whole number
answer_29_R3c = ...  # c) the letter of your choice, e.g. "B"
answer_29_R3d = ...  # d) True or False
answer_29_R3e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_29_R3a, answer_29_R3b, answer_29_R3c, answer_29_R3d, answer_29_R3e]):
    wb.check(f"29.R3{letter}", answer)

**Ex 29.1 — Poids en mémoire de MNIST, Fashion-MNIST et CIFAR-10**

In [ ]:
# Ex 29.1: the values you found on paper
answer_29_1a = ...  # a) a number, 2 decimals
answer_29_1b = ...  # b) a number, 2 decimals
answer_29_1c = ...  # c) a number, 2 decimals
answer_29_1d = ...  # d) a number, 3 decimals
answer_29_1e = ...  # e) a whole number
answer_29_1f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_29_1a, answer_29_1b, answer_29_1c, answer_29_1d, answer_29_1e, answer_29_1f]):
    wb.check(f"29.1{letter}", answer)

**Ex 29.2 — Découpage stratifié et référence « classe majoritaire »**

In [ ]:
# Ex 29.2: the values you found on paper
answer_29_2a = ...  # a) a whole number
answer_29_2b = ...  # b) a whole number
answer_29_2c = ...  # c) a number, 2 decimals
answer_29_2d = ...  # d) a number, 2 decimals
answer_29_2e = ...  # e) a number, 2 decimals
answer_29_2f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_29_2a, answer_29_2b, answer_29_2c, answer_29_2d, answer_29_2e, answer_29_2f]):
    wb.check(f"29.2{letter}", answer)

## Outils du notebook

La cellule suivante importe pandas et PyTorch et définit les petits outils des deux parties : `train_mlp` et `evaluate` (le petit réseau de 29.8, 29.9 et 29.14), `fine_tune`, `digits_naive`, `show_digits`, `online`, un Hub et un OpenML simulés pour 29.11 (`FakeHubApi`, `fake_fetch_openml`) et `check_card` pour la data card de 29.15. Lis leurs docstrings au besoin.

In [ ]:
# Tools for the notebook exercises (parts A and B)
import copy
import hashlib
import re
import socket
import tempfile
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wb.nbtools import StopCheck, checked, filled, fr, returned, same, verdict

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:                       # a computer without PyTorch (Mac Intel): parts of this chapter need Colab
    torch = None
    print("⏳ PyTorch absent sur cet ordinateur : 29.8, 29.9 et 29.14 se font sur Colab (00_setup/INSTALL_LOCAL.md).")

WORK_DIR = Path(tempfile.mkdtemp(prefix="wb_ch29_"))   # a temporary folder for the files the exercises write


def pct(x, digits=1):
    """A share written the French way: 0.734 -> '73,4 %'."""
    return f"{100 * x:.{digits}f}".replace(".", ",") + " %"


def online(host="huggingface.co", timeout=5.0):
    """True if `host` answers on port 443 (the live parts of 29.10 and 29.11 are skipped without a network)."""
    try:
        socket.create_connection((host, 443), timeout=timeout).close()
        return True
    except OSError:
        return False


def train_mlp(X, y, epochs=2, seed=0, batch_size=64, lr=1e-3):
    """The small network of 29.8, 29.9 and 29.14: 784-128-10 (ReLU), `epochs` epochs of Adam on uint8 images
    (N, 28, 28) divided by 255. Deterministic for a given seed."""
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, yt = torch.as_tensor(X, dtype=torch.float32) / 255.0, torch.as_tensor(y, dtype=torch.long)
    generator = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        order = torch.randperm(len(Xt), generator=generator)
        for i in range(0, len(Xt), batch_size):
            idx = order[i:i + batch_size]
            loss = F.cross_entropy(model(Xt[idx]), yt[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model.eval()


def evaluate(model, X, y):
    """(accuracy, mean confidence) of `model` on uint8 images X (N, 28, 28): the confidence of one image is its
    largest softmax probability."""
    with torch.no_grad():
        logits = model(torch.as_tensor(X, dtype=torch.float32) / 255.0)
    probs = F.softmax(logits, dim=1)
    return float((logits.argmax(1).numpy() == np.asarray(y)).mean()), float(probs.max(1).values.mean())


def fine_tune(model, X, y, steps=100, lr=1e-3):
    """A copy of `model` trained `steps` more steps of Adam on all of (X, y) at once (a few labelled examples)."""
    model = copy.deepcopy(model).train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, yt = torch.as_tensor(X, dtype=torch.float32) / 255.0, torch.as_tensor(y, dtype=torch.long)
    for _ in range(steps):
        loss = F.cross_entropy(model(Xt), yt)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model.eval()


def digits_naive(images):
    """The 8 x 8 digits of scikit-learn (values 0 to 16) enlarged as they are to 28 x 28, uint8 0 to 255 (29.9)."""
    t = torch.as_tensor(np.asarray(images), dtype=torch.float32)[:, None] / 16.0
    out = F.interpolate(t, size=(28, 28), mode="bilinear", align_corners=False)[:, 0] * 255.0
    return np.clip(np.rint(out.numpy()), 0, 255).astype(np.uint8)


def show_digits(rows, titles, n=8):
    """One row of n images (28 x 28) per array of `rows`, its title on the left."""
    fig, axes = plt.subplots(len(rows), n, figsize=(1.1 * n, 1.25 * len(rows)))
    axes = np.atleast_2d(axes)
    for r, (images, title) in enumerate(zip(rows, titles)):
        for c in range(n):
            axes[r, c].imshow(images[c], cmap="gray", vmin=0, vmax=255)
            axes[r, c].axis("off")
        axes[r, 0].set_title(title, fontsize=8, loc="left")
    plt.tight_layout()
    plt.show()


# 29.11: stand-ins for the Hugging Face Hub and OpenML, so that the checks never depend on the network
FAKE_HUB = [SimpleNamespace(id=i, downloads=d, tags=t) for i, d, t in [
    ("demo/penguins-clean", 950, ["task_categories:tabular-classification", "license:cc0-1.0"]),
    ("demo/penguins-raw", 120, ["license:cc-by-4.0"]),
    ("someone/penguins-images", 3400, ["license:cc-by-nc-4.0", "size_categories:1K<n<10K"]),
    ("other/penguin-sounds", 40, []),
    ("lab/penguins-gps", 610, ["license:odbl"]),
    ("demo/fruits", 5000, ["license:mit"]),
]]


class FakeHubApi:
    """Answers list_datasets(search=..., limit=..., sort="downloads") like HfApi, from FAKE_HUB (no network)."""
    def __init__(self):
        self.calls = []

    def list_datasets(self, search=None, limit=None, sort=None, **kwargs):
        self.calls.append({"search": search, "limit": limit, "sort": sort, **kwargs})
        rows = [r for r in FAKE_HUB if search is None or search.lower() in r.id.lower()]
        if sort == "downloads":
            rows = sorted(rows, key=lambda r: -r.downloads)
        return iter(rows[:limit] if limit else rows)


def fake_fetch_openml(data_id=None, as_frame=True, **kwargs):
    """Answers like sklearn.datasets.fetch_openml(data_id=42585, as_frame=True), from the workbook copy of Penguins."""
    from sklearn.utils import Bunch
    frame = wb.datasets.load_penguins().drop(columns=["year"], errors="ignore")
    details = {"id": str(data_id), "name": "penguins", "version": "1", "licence": "CC0",
               "default_target_attribute": "species", "original_data_url": "https://github.com/allisonhorst/palmerpenguins",
               "md5_checksum": "0d4936a104bb85ef5127a5a63dce81f9"}
    return Bunch(frame=frame, details=details, target_names=["species"])


# 29.15: the sections a data card must have (the sheet, « Au-delà du livre : documenter et télécharger »)
CARD_SECTIONS = ["Provenance", "Contenu", "Licence", "Usage prévu", "Limites et biais", "Téléchargement"]


def check_card(text):
    """The list of problems of a data card written in Markdown (empty when it is complete)."""
    problems = [f"section « ## {s} » absente" for s in CARD_SECTIONS
                if not re.search(rf"^##\s+{re.escape(s)}\s*$", text, flags=re.M)]
    if "..." in text:
        problems.append("il reste des « ... » à remplacer")
    if not re.search(r"\b[0-9a-f]{64}\b", text):
        problems.append("aucune somme de contrôle SHA-256 (64 caractères hexadécimaux) dans la carte")
    if not re.search(r"https?://", text):
        problems.append("aucune adresse (URL) de la source")
    return problems

## Partie A · Charger, chercher, comparer

Les datasets intégrés aux bibliothèques, deux sources d'images au même format, le Hugging Face Hub et OpenML. Chaque fois, la même discipline : mesurer sur les données chargées plutôt que croire la description, et noter d'où viennent les données et sous quelle licence.

### Ex 29.7 — Les datasets intégrés de scikit-learn, et une affirmation du livre à vérifier 📦 ★ ⏱️ 15 min
**Objectif :** inventorier les datasets intégrés de scikit-learn en mesurant les données chargées, et vérifier qu'un loader existe encore.  
**Prérequis :** ch. 15 (scikit-learn) · fiche §29.3.1 (encadrés ⚠️ et 🕰️) · **Parcours :** R, M, C

Le livre annonce 5 620 images pour Digits et liste Boston parmi les datasets de scikit-learn. Vérifie sur ta version.

**Contrat**
- `inventory_7()` : un `DataFrame`, une ligne par dataset de `SKLEARN_LOADERS_7` (dans cet ordre), colonnes `"dataset"`, `"exemples"`, `"variables"`, `"tache"` (`"classification"` ou `"régression"`, d'après `REGRESSION_7`) et `"classes"` (le nombre de labels distincts, 0 pour une régression). Tout se **mesure** sur `X, y = getattr(skd, loader)(return_X_y=True)`.
- `has_loader_7(name)` : `True` si `sklearn.datasets` fournit encore la fonction `name`, `False` sinon, sans lever d'erreur. Piège : pour un loader retiré, scikit-learn lève une `ImportError` explicative, pas une `AttributeError` ; `hasattr` ne suffit donc pas.

La cellule de vérification compare ta table aux mesures, puis te fait reporter :
a) le nombre d'images de Digits, mesuré par ta table ;
b) la part de la classe minoritaire de Breast Cancer, à 3 décimales (`minority_7`, à calculer) ;
c) ce que rend `has_loader_7("load_boston")`.

Dans tes notes : lis le message de l'`ImportError` de `load_boston` (`try: skd.load_boston() except ImportError as e: print(e)`). Que propose scikit-learn à la place ? Pourquoi une table mesurée vaut-elle mieux qu'une liste recopiée d'un livre ou d'une page web ?

In [ ]:
from sklearn import datasets as skd

# The loaders to inventory (name -> function of sklearn.datasets); "diabetes" is a regression
SKLEARN_LOADERS_7 = {"iris": "load_iris", "digits": "load_digits", "wine": "load_wine",
                     "breast_cancer": "load_breast_cancer", "diabetes": "load_diabetes"}
REGRESSION_7 = {"diabetes"}

In [ ]:
def inventory_7():
    """A DataFrame, one row per dataset of SKLEARN_LOADERS_7 (in this order), with the columns "dataset" (its name),
    "exemples", "variables", "tache" ("classification" or "régression", from REGRESSION_7) and "classes" (the number
    of distinct labels, 0 for a regression), measured on the loaded data (getattr(skd, loader)(return_X_y=True))."""
    raise NotImplementedError("inventory_7")


def has_loader_7(name):
    """True if sklearn.datasets still provides the function `name` (e.g. "load_iris"), False otherwise. Careful: for
    a loader that was removed, scikit-learn raises an ImportError (with an explanation), not an AttributeError."""
    raise NotImplementedError("has_loader_7")


minority_7 = ...   # b) the share of the minority class of breast_cancer, 3 decimals

In [ ]:
with checked("29.7"):
    table_7 = inventory_7()
    if not returned("29.7", "inventory_7", table_7):
        raise StopCheck
    columns_7 = ["dataset", "exemples", "variables", "tache", "classes"]
    verdict("29.7", isinstance(table_7, pd.DataFrame) and list(table_7.columns) == columns_7,
            "un DataFrame aux cinq colonnes demandées.",
            f"attendu un DataFrame aux colonnes {columns_7}, reçu {list(getattr(table_7, 'columns', []))}.")
    want_7 = []
    for name_7, loader_7 in SKLEARN_LOADERS_7.items():
        X_w7, y_w7 = getattr(skd, loader_7)(return_X_y=True)
        reg_7 = name_7 in REGRESSION_7
        want_7.append([name_7, X_w7.shape[0], X_w7.shape[1], "régression" if reg_7 else "classification",
                       0 if reg_7 else len(np.unique(y_w7))])
    got_7 = table_7[columns_7].values.tolist() if list(table_7.columns) == columns_7 else []
    bad_7 = [w[0] for w, g in zip(want_7, got_7) if [str(v) for v in w] != [str(v) for v in g]]
    verdict("29.7", len(got_7) == len(want_7) and not bad_7, "les tailles, les tâches et les classes sont justes.",
            f"lignes fausses ou manquantes : {bad_7 or 'nombre de lignes'} (mesure sur X.shape et np.unique(y)).")
    verdict("29.7", has_loader_7("load_iris") is True and has_loader_7("fetch_openml") is True,
            "has_loader_7 reconnaît les fonctions présentes.", "has_loader_7('load_iris') devrait rendre True.")
    verdict("29.7", has_loader_7("load_nothing_7") is False, "has_loader_7 rend False pour un nom inconnu.",
            "has_loader_7('load_nothing_7') devrait rendre False, sans lever d'erreur.")
    wb.check("29.7a", int(table_7.set_index("dataset").loc["digits", "exemples"]), computed=True)
    wb.check("29.7c", has_loader_7("load_boston"), computed=True)
wb.check("29.7b", minority_7)

### Ex 29.8 — Prédire : le même modèle sur MNIST puis sur Fashion-MNIST 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que coûte le passage de MNIST à Fashion-MNIST à un même petit réseau, et ce que fait un modèle de chiffres face à des vêtements.  
**Prérequis :** Ex 29.7 · fiche §29.2 · **Parcours :** R, M, C

La cellule suivante entraîne deux fois le même petit réseau dense (784-128-10, deux epochs d'Adam sur 10 000 images, `train_mlp` dans les outils) : `MODEL_M` sur MNIST, `MODEL_F` sur Fashion-MNIST. Même format, même code, seule la source change. Elle n'affiche rien.

**Avant d'exécuter l'expérience**, écris tes trois prédictions :
a) l'accuracy de test de `MODEL_M` sur MNIST : `"A"` moins de 0,90, `"B"` de 0,90 à 0,97, `"C"` plus de 0,97 ;
b) les points d'accuracy perdus par `MODEL_F` sur Fashion-MNIST par rapport à a) : `"A"` moins de 5, `"B"` de 5 à 15, `"C"` plus de 15 ;
c) l'accuracy de `MODEL_M` (entraîné sur des chiffres) sur le test de Fashion-MNIST : `"A"` moins de 0,2, `"B"` de 0,2 à 0,5, `"C"` plus de 0,5.

La cellule d'expérience n'affiche rien tant que tes trois prédictions ne sont pas écrites.

In [ ]:
# The same small network (784-128-10, two epochs of Adam on 10 000 training images, train_mlp in the tools) trained
# twice: on MNIST, then on Fashion-MNIST. Nothing is printed here: the accuracies come after your predictions.
XM_TRAIN, YM_TRAIN = wb.datasets.load_mnist("train", 10_000)
XM_TEST, YM_TEST = wb.datasets.load_mnist("test")
XF_TRAIN, YF_TRAIN = wb.datasets.load_fashion_mnist("train", 10_000)
XF_TEST, YF_TEST = wb.datasets.load_fashion_mnist("test")
MODEL_M = train_mlp(XM_TRAIN, YM_TRAIN) if torch is not None else None
MODEL_F = train_mlp(XF_TRAIN, YF_TRAIN) if torch is not None else None

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
mnist_8 = ...     # a) the MNIST test accuracy of MODEL_M: "A" below 0.90, "B" from 0.90 to 0.97, "C" above 0.97
drop_8 = ...      # b) the points lost by MODEL_F on Fashion-MNIST: "A" fewer than 5, "B" from 5 to 15, "C" more than 15
crossed_8 = ...   # c) MODEL_M (trained on digits) on the Fashion-MNIST test: "A" below 0.2, "B" 0.2 to 0.5, "C" above 0.5

In [ ]:
if any(answer is ... or answer is None for answer in [mnist_8, drop_8, crossed_8]):
    print("⏳ Ex 29.8 : écris d'abord tes trois prédictions dans la cellule précédente.")
else:
    acc_m8, conf_m8 = evaluate(MODEL_M, XM_TEST, YM_TEST)
    acc_f8, conf_f8 = evaluate(MODEL_F, XF_TEST, YF_TEST)
    acc_x8, conf_x8 = evaluate(MODEL_M, XF_TEST, YF_TEST)
    print(f"MNIST : accuracy {fr(acc_m8)} ; Fashion-MNIST : accuracy {fr(acc_f8)}, soit {fr(100 * (acc_m8 - acc_f8), 1)} points de moins")
    print(f"Le modèle MNIST sur Fashion-MNIST : accuracy {fr(acc_x8)} (le hasard : 0,1), confiance moyenne {fr(conf_x8)}")
    with torch.no_grad():
        pred_f8 = MODEL_F(torch.as_tensor(XF_TEST, dtype=torch.float32) / 255.0).argmax(1).numpy()
    recall_f8 = [float((pred_f8[YF_TEST == c] == c).mean()) for c in range(10)]
    plt.figure(figsize=(8, 3))
    plt.bar(wb.datasets.FASHION_CLASSES, recall_f8, color="tab:orange")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("recall")
    plt.title("Fashion-MNIST : recall de chaque classe")
    plt.tight_layout()
    plt.show()
    wb.check("29.8a", mnist_8)
    wb.check("29.8b", drop_8)
    wb.check("29.8c", crossed_8)

Dans tes notes : quelles classes de Fashion-MNIST le réseau reconnaît-il le moins bien (le graphique), et pourquoi (Q4 e) ? Que dit c) de la confiance d'un modèle hors de sa distribution d'entraînement ? Pourquoi le livre conseille-t-il Fashion-MNIST comme étape après MNIST ?

### Ex 29.9 — Prédire la chute d'accuracy en changeant de source de chiffres 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que devient un modèle MNIST face à d'autres chiffres manuscrits, et s'il s'en rend compte.  
**Prérequis :** Ex 29.8 · fiche §29.2, « Au-delà du livre : le décalage de distribution » · **Parcours :** R, M, C

scikit-learn fournit d'autres chiffres manuscrits : 1 797 images de 8 × 8 pixels, valeurs de 0 à 16 (`load_digits`). La cellule suivante les agrandit **telles quelles** à 28 × 28, valeurs de 0 à 255 (`digits_naive`, dans les outils) : même format que MNIST, mêmes dix classes. On les donne à `MODEL_M` (29.8), sans autre entraînement.

**Avant d'exécuter**, écris tes deux prédictions :
a) l'accuracy de `MODEL_M` sur ces chiffres : `"A"` plus de 0,7, `"B"` de 0,4 à 0,7, `"C"` moins de 0,4 ;
b) la **confiance moyenne** du modèle sur ces chiffres (pour chaque image, sa plus grande probabilité softmax ; puis la moyenne) reste-t-elle au-dessus de 0,6 ? `True` ou `False`.

La cellule d'expérience montre aussi quelques images des deux sources côte à côte, et n'affiche rien tant que tes deux prédictions ne sont pas écrites.

In [ ]:
from sklearn.datasets import load_digits

DIGITS_9 = load_digits()                       # 1 797 digits of 8 x 8 pixels, values 0 to 16 (an UCI collection)
XD_9 = digits_naive(DIGITS_9.images)           # enlarged as they are to 28 x 28, 0 to 255: the MNIST format?
YD_9 = DIGITS_9.target

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
digits_9 = ...      # a) the accuracy of MODEL_M on XD_9: "A" above 0.7, "B" from 0.4 to 0.7, "C" below 0.4
confident_9 = ...   # b) True if the mean confidence of MODEL_M on XD_9 stays above 0.6, else False

In [ ]:
if any(answer is ... or answer is None for answer in [digits_9, confident_9]):
    print("⏳ Ex 29.9 : écris d'abord tes deux prédictions dans la cellule précédente.")
else:
    show_digits([XM_TEST[:8], XD_9[:8]], ["MNIST (test)", "scikit-learn, agrandis tels quels"])
    acc_m9, conf_m9 = evaluate(MODEL_M, XM_TEST, YM_TEST)
    acc_d9, conf_d9 = evaluate(MODEL_M, XD_9, YD_9)
    print(f"MNIST (test)          : accuracy {fr(acc_m9)}, confiance moyenne {fr(conf_m9)}")
    print(f"chiffres scikit-learn : accuracy {fr(acc_d9)}, confiance moyenne {fr(conf_d9)}")
    print(f"pixels moyens : {fr(float(XM_TEST.mean()), 1)} sur MNIST, {fr(float(XD_9.mean()), 1)} ici (0 à 255)")
    wb.check("29.9a", digits_9)
    wb.check("29.9b", confident_9)

Dans tes notes : compare les deux rangées d'images. Qu'est-ce qui diffère, à part la taille d'origine (la place du chiffre dans l'image, l'épaisseur du trait, le flou) ? Pourquoi une confiance élevée sur des données décalées est-elle dangereuse en production ? 29.14 mesure ce que l'on récupère en préparant mieux les images.

### Ex 29.10 — torchvision.datasets et Hugging Face datasets : mêmes données, deux API 📦 ★★ ⏱️ 25 min
**Objectif :** charger un dataset du Hugging Face Hub à une révision fixée, le convertir en tableaux NumPy et le comparer à une autre source.  
**Prérequis :** ch. 23 (Dataset, DataLoader) · fiche §29.3.2 (encadré 🕰️, encadré 🧮) · **Parcours :** R, M, C

La cellule suivante charge le split de test de Fashion-MNIST depuis le Hugging Face Hub, à une **révision** fixée (un commit du dépôt du dataset) : `load_dataset(nom, split="test", revision=...)`. Sans réseau, elle fabrique le même format depuis la copie du workbook et le dit. `wb.datasets.load_fashion_mnist`, lui, passe par `torchvision.datasets.FashionMNIST` (ou un miroir) et rend des tableaux NumPy.

**Contrat**
- `hf_to_numpy_10(split)` : depuis un split aux colonnes `"image"` (images PIL) et `"label"`, rend `(X, y)` : `X` en `uint8` de forme `(N, 28, 28)`, `y` en `int64` de forme `(N,)`, dans l'ordre du split ;
- `compare_10(Xa, ya, Xb, yb)` : un dictionnaire `{"same_shape", "same_labels", "same_pixels", "max_diff"}` ; `max_diff` est le plus grand écart absolu entre deux pixels (0 si identiques, −1 si les formes diffèrent). Piège : en `uint8`, `3 - 10` vaut 249.

La cellule de vérification teste `compare_10` sur des cas fabriqués (un pixel modifié, des labels mélangés, des formes différentes), puis compare les deux sources de Fashion-MNIST et affiche les noms des classes de chacune.

Dans tes notes : les deux sources sont-elles identiques, pixel à pixel et dans le même ordre ? Les noms des classes sont-ils écrits pareil ? Pourquoi fixer `revision=` plutôt que prendre la dernière version ? Écris l'équivalent `torchvision.datasets.FashionMNIST(root, train=False, download=True)` et dis ce qu'il rend (des images PIL et des entiers, transformables par `transform=`).

In [ ]:
# One version of Fashion-MNIST on the Hugging Face Hub, pinned to an exact revision (a commit of the dataset repository)
FASHION_REPO_10 = "zalando-datasets/fashion_mnist"
FASHION_REV_10 = "531be5e2ccc9dba0c201ad3ae567a4f3d16ecdd2"
import warnings
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")              # tqdm's notice about ipywidgets, printed once at import
        from datasets import Dataset, load_dataset
    try:
        with wb.datasets.quiet_huggingface():
            HF_10 = load_dataset(FASHION_REPO_10, split="test", revision=FASHION_REV_10,
                                 cache_dir=str(wb.datasets.cache_dir() / "hf"))
        SOURCE_10 = "le Hugging Face Hub"
    except Exception as exc:              # no network: the same format, built from the workbook copy
        from PIL import Image
        X_off10, y_off10 = wb.datasets.load_fashion_mnist("test")
        HF_10 = Dataset.from_dict({"image": [Image.fromarray(img) for img in X_off10], "label": y_off10.tolist()})
        SOURCE_10 = f"une copie locale au même format (Hub injoignable : {type(exc).__name__})"
except ImportError:
    HF_10, SOURCE_10 = None, "rien : la bibliothèque datasets est absente (pip install datasets)"
print(f"Fashion-MNIST (test) chargé depuis {SOURCE_10} : {HF_10}")

In [ ]:
def hf_to_numpy_10(split):
    """(X, y) from a split of Hugging Face `datasets` with the columns "image" (PIL images) and "label": X uint8
    (N, 28, 28) and y int64 (N,), in the order of the split."""
    raise NotImplementedError("hf_to_numpy_10")


def compare_10(Xa, ya, Xb, yb):
    """Compare two versions of the same images: a dict with "same_shape" (bool), "same_labels" (bool: the same labels
    in the same order), "same_pixels" (bool) and "max_diff" (int: the largest absolute difference between two pixels,
    0 when identical, -1 when the shapes differ)."""
    raise NotImplementedError("compare_10")

In [ ]:
with checked("29.10"):
    if HF_10 is None:
        print("⏳ Ex 29.10 : la bibliothèque datasets est absente : pip install datasets (déjà là sur Colab).")
        raise StopCheck
    X_hf10, y_hf10 = hf_to_numpy_10(HF_10)
    if not returned("29.10", "hf_to_numpy_10", X_hf10):
        raise StopCheck
    verdict("29.10", isinstance(X_hf10, np.ndarray) and X_hf10.shape == (len(HF_10), 28, 28) and X_hf10.dtype == np.uint8,
            "X : uint8 de forme (N, 28, 28).", f"X attendu uint8 (N, 28, 28), reçu {getattr(X_hf10, 'dtype', type(X_hf10))} {getattr(X_hf10, 'shape', '')}.")
    verdict("29.10", np.asarray(y_hf10).dtype == np.int64 and np.asarray(y_hf10).shape == (len(HF_10),),
            "y : int64 de forme (N,).", f"y attendu int64 (N,), reçu {np.asarray(y_hf10).dtype} {np.asarray(y_hf10).shape}.")
    verdict("29.10", int(X_hf10[0].sum()) == int(np.asarray(HF_10[0]["image"], dtype=np.int64).sum()) and int(y_hf10[0]) == HF_10[0]["label"],
            "le premier exemple est bien le premier du split.", "le premier exemple ne correspond pas à HF_10[0] : garde l'ordre du split.")
    X_wb10, y_wb10 = wb.datasets.load_fashion_mnist("test")
    rng_10 = np.random.default_rng(10)
    X_a10 = X_wb10[:50].copy()
    r_10, c_10 = np.unravel_index(int(X_a10[3].argmax()), (28, 28))
    X_a10[3, r_10, c_10] -= 7                                           # its brightest pixel, darker by 7
    probe_10 = compare_10(X_wb10[:50], y_wb10[:50], X_a10, y_wb10[:50])
    if not returned("29.10", "compare_10", probe_10):
        raise StopCheck
    keys_10 = {"same_shape", "same_labels", "same_pixels", "max_diff"}
    verdict("29.10", isinstance(probe_10, dict) and set(probe_10) == keys_10, "compare_10 rend un dict aux quatre clés.",
            f"attendu un dict aux clés {sorted(keys_10)}, reçu {probe_10!r:.120}.")
    verdict("29.10", probe_10.get("max_diff") == 7 and probe_10.get("same_pixels") is False,
            "un pixel plus sombre de 7 est vu (max_diff = 7).",
            f"un pixel plus sombre de 7 : attendu max_diff = 7, reçu {probe_10.get('max_diff')} (en uint8, 3 − 10 vaut 249 : "
            "convertis en entiers signés avant de soustraire).")
    shuffled_10 = compare_10(X_wb10[:50], y_wb10[:50], X_wb10[:50], rng_10.permutation(y_wb10[:50]))
    verdict("29.10", shuffled_10.get("same_labels") is False and shuffled_10.get("same_pixels") is True,
            "des labels mélangés sont vus.", "des labels mélangés (images identiques) : attendu same_labels False, same_pixels True.")
    other_10 = compare_10(X_wb10[:50], y_wb10[:50], X_wb10[:40], y_wb10[:40])
    verdict("29.10", other_10.get("same_shape") is False and other_10.get("max_diff") == -1,
            "des formes différentes donnent max_diff = -1, sans erreur.", f"formes différentes : attendu same_shape False et max_diff -1, reçu {other_10}.")
    result_10 = compare_10(X_hf10, y_hf10, X_wb10, y_wb10)
    print(f"Hub (révision {FASHION_REV_10[:8]}) contre wb.datasets.load_fashion_mnist('test') : {result_10}")
    print("Noms des classes sur le Hub :", HF_10.features["label"].names if hasattr(HF_10.features["label"], "names") else "(pas de noms)")
    print("Noms des classes du workbook :", wb.datasets.FASHION_CLASSES)

### Ex 29.11 — Explorer le Hugging Face Hub et OpenML : chercher, filtrer, lire la licence 📦 ★★ ⏱️ 25 min
**Objectif :** chercher des datasets par programme sur le Hugging Face Hub et OpenML, et en lire la licence, la taille et la source.  
**Prérequis :** Ex 29.10 · fiche §29.4 (encadré 🕰️), §29.5 · **Parcours :** R, M, C

**Contrat**
- `search_hub_11(query, limit=5, api=None)` : les `limit` datasets du Hub **les plus téléchargés** que trouve `query`, du plus au moins téléchargé, dans un `DataFrame` aux colonnes `"id"`, `"downloads"` et `"licence"`. La licence vient d'un tag `"license:xxx"` (`"inconnue"` sans tag). `api` est un `HfApi` (un nouveau si `None`) : `api.list_datasets(search=..., limit=..., sort="downloads")` rend des objets qui ont `.id`, `.downloads` et `.tags`.
- `openml_summary_11(data_id, fetch=None)` : un dictionnaire sur un dataset d'OpenML, lu dans `bunch = fetch(data_id=data_id, as_frame=True)` (`fetch` vaut par défaut `sklearn.datasets.fetch_openml`) : `"name"`, `"version"`, `"licence"` (dans `bunch.details`), `"rows"` et `"columns"` (de `bunch.frame`, cible comprise), `"target"` (`details["default_target_attribute"]`) et `"source"` (`details["original_data_url"]`, `None` s'il manque).

La cellule de vérification ne dépend pas du réseau : elle passe à tes fonctions un Hub et un OpenML **simulés** (`FakeHubApi`, `fake_fetch_openml`, dans les outils). La cellule suivante les lance pour de vrai si le réseau répond : la recherche `"penguins"` sur le Hub, puis Palmer Penguins sur OpenML (`data_id=42585`).

Dans tes notes : parmi les résultats du Hub, lesquels pourrais-tu utiliser dans un produit, d'après leur licence ? Que fais-tu d'un dataset sans licence ? Ouvre la page (la *dataset card*) du premier résultat : que dit-elle de la provenance des données ? Cherche deux datasets candidats pour ton projet final et note leur licence et leur taille.

In [ ]:
def search_hub_11(query, limit=5, api=None):
    """A DataFrame of the `limit` most downloaded datasets of the Hugging Face Hub found by `query`, the most downloaded
    first, with the columns "id", "downloads" and "licence" (taken from a tag "license:xxx", "inconnue" without one).
    `api` is an HfApi (a new one when None): use api.list_datasets(search=..., limit=..., sort="downloads")."""
    raise NotImplementedError("search_hub_11")


def openml_summary_11(data_id, fetch=None):
    """A dict about a dataset of OpenML, from fetch(data_id=data_id, as_frame=True) (fetch: by default
    sklearn.datasets.fetch_openml): "name", "version", "licence", "rows" and "columns" (of bunch.frame, target
    included), "target" (details["default_target_attribute"]) and "source" (details["original_data_url"], None when
    absent)."""
    raise NotImplementedError("openml_summary_11")

In [ ]:
with checked("29.11"):
    api_11 = FakeHubApi()
    found_11 = search_hub_11("penguin", limit=3, api=api_11)
    if not returned("29.11", "search_hub_11", found_11):
        raise StopCheck
    verdict("29.11", isinstance(found_11, pd.DataFrame) and list(found_11.columns) == ["id", "downloads", "licence"],
            "search_hub_11 rend un DataFrame aux colonnes id, downloads, licence.",
            f"attendu les colonnes ['id', 'downloads', 'licence'], reçu {list(getattr(found_11, 'columns', []))}.")
    want_11 = [("someone/penguins-images", 3400, "cc-by-nc-4.0"), ("demo/penguins-clean", 950, "cc0-1.0"),
               ("lab/penguins-gps", 610, "odbl")]
    got_11 = [tuple(r) for r in found_11[["id", "downloads", "licence"]].itertuples(index=False)] if len(found_11.columns) == 3 else []
    verdict("29.11", got_11 == want_11, "les 3 plus téléchargés, dans l'ordre, avec leur licence (sur un Hub simulé).",
            f"sur le Hub simulé, attendu {want_11}, reçu {got_11} (demande sort=\"downloads\" et limit à list_datasets).")
    none_11 = search_hub_11("sounds", limit=5, api=FakeHubApi())
    verdict("29.11", list(none_11["licence"]) == ["inconnue"], "un dataset sans tag de licence : « inconnue ».",
            f"pour un dataset sans tag license:, attendu « inconnue », reçu {list(none_11['licence'])}.")
    summary_11 = openml_summary_11(42585, fetch=fake_fetch_openml)
    if not returned("29.11", "openml_summary_11", summary_11):
        raise StopCheck
    frame_11 = fake_fetch_openml(42585).frame
    want_s11 = {"name": "penguins", "version": "1", "licence": "CC0", "rows": frame_11.shape[0],
                "columns": frame_11.shape[1], "target": "species", "source": "https://github.com/allisonhorst/palmerpenguins"}
    verdict("29.11", summary_11 == want_s11, "openml_summary_11 lit le nom, la version, la licence, la taille, la cible et la source.",
            f"sur un OpenML simulé, attendu {want_s11}, reçu {summary_11}.")

In [ ]:
# Live, if the network answers (a few seconds; OpenML caches its files in the workbook cache)
if online("huggingface.co"):
    with checked("29.11"):
        print(search_hub_11("penguins", limit=5).to_string(index=False))
else:
    print("⏳ Hub injoignable : la recherche en direct est sautée (les vérifications ci-dessus suffisent).")
if online("www.openml.org"):
    with checked("29.11"):
        from sklearn.datasets import fetch_openml
        def fetch_cached_11(**kwargs):
            return fetch_openml(data_home=str(wb.datasets.cache_dir() / "openml"), **kwargs)
        print(openml_summary_11(42585, fetch=fetch_cached_11))
else:
    print("⏳ OpenML injoignable : le résumé en direct est sauté.")

## Partie B · Auditer, documenter, déjouer les pièges

Avant d'entraîner quoi que ce soit : profiler un dataset inconnu, traquer les doublons qui gonflent un score, mesurer un décalage de distribution, puis écrire la data card et le script de téléchargement de ton projet final. Le défi final rassemble tout sur un tableau qui cache huit défauts.

### Ex 29.12 — Profil express d'un dataset inconnu 📈 ★★ ⏱️ 20 min
**Objectif :** écrire en quelques lignes le profil de chaque colonne d'un tableau, et y lire ses défauts.  
**Prérequis :** ch. 12 (pandas) · 0A · fiche « Au-delà du livre : auditer un dataset » · **Parcours :** R, M, C

Adult (le recensement américain de 1994, `data/cards/adult.md`) est chargé dans `ADULT_12`.

**Contrat** de `profile_12(df)` : un `DataFrame` indexé par les noms des colonnes de `df` (dans leur ordre), aux colonnes `"type"` (`str` du dtype), `"manquants"` (le nombre de valeurs manquantes), `"part_manquants"` (leur part, entre 0 et 1), `"distinctes"` (le nombre de valeurs distinctes, sans les manquantes), `"mode"` (la valeur la plus fréquente) et `"part_mode"` (sa part parmi les valeurs présentes).

La cellule de vérification compare ton profil aux mesures de pandas et trace deux graphiques (la part de manquants et la part du mode, colonne par colonne). Reporte :
a) la colonne d'`ADULT_12` qui a le plus de valeurs manquantes ;
b) le nombre de lignes qui répètent une ligne précédente (`duplicated`) ;
c) la part de la classe majoritaire de la colonne `"class"`, à 2 décimales.

Dans tes notes : quelles colonnes sont presque constantes (une part du mode au-dessus de 0,9) ? Sont-elles inutiles pour autant ? Les doublons de b) sont-ils des erreurs, ou deux personnes qui se ressemblent ? Que vaut un modèle qui prédit toujours la classe majoritaire (✏️ 29.2) ?

In [ ]:
ADULT_12 = wb.datasets.load_adult()     # the 1994 US census (48 842 people), data/cards/adult.md
ADULT_12.head(3)

In [ ]:
def profile_12(df):
    """One row per column of df (same order, index = the column names), with the columns "type" (str of the dtype),
    "manquants" (the number of missing values), "part_manquants" (their share, 0 to 1), "distinctes" (the number of
    distinct values, missing ones excluded), "mode" (the most frequent value) and "part_mode" (its share among the
    values that are not missing)."""
    raise NotImplementedError("profile_12")


most_missing_12 = ...   # a) the column of ADULT_12 with the most missing values (its name, between quotes)
duplicates_12 = ...     # b) the number of rows of ADULT_12 that repeat an earlier row
majority_12 = ...       # c) the share of the majority class of the column "class", 2 decimals

In [ ]:
with checked("29.12"):
    prof_c12 = profile_12(ADULT_12)
    if not returned("29.12", "profile_12", prof_c12):
        raise StopCheck
    cols_12 = ["type", "manquants", "part_manquants", "distinctes", "mode", "part_mode"]
    verdict("29.12", isinstance(prof_c12, pd.DataFrame) and list(prof_c12.columns) == cols_12
            and list(prof_c12.index) == list(ADULT_12.columns),
            "une ligne par colonne, les six colonnes demandées.",
            f"attendu l'index {list(ADULT_12.columns)[:3]}… et les colonnes {cols_12}, reçu {list(getattr(prof_c12, 'columns', []))}.")
    same("29.12", "manquants", prof_c12["manquants"].to_numpy(dtype=float), ADULT_12.isna().sum().to_numpy(dtype=float))
    same("29.12", "part_manquants", prof_c12["part_manquants"].to_numpy(dtype=float), ADULT_12.isna().mean().to_numpy(dtype=float))
    same("29.12", "distinctes", prof_c12["distinctes"].to_numpy(dtype=float), ADULT_12.nunique().to_numpy(dtype=float))
    want_mode_12 = [float(ADULT_12[c].value_counts(normalize=True).iloc[0]) for c in ADULT_12.columns]
    same("29.12", "part_mode", prof_c12["part_mode"].to_numpy(dtype=float), np.array(want_mode_12), rtol=1e-9)
    verdict("29.12", str(prof_c12.loc["sex", "mode"]) == str(ADULT_12["sex"].mode().iloc[0]), "le mode est la valeur la plus fréquente.",
            f"mode de « sex » : attendu {ADULT_12['sex'].mode().iloc[0]!r}, reçu {prof_c12.loc['sex', 'mode']!r}.")
    fig_12, axes_12 = plt.subplots(1, 2, figsize=(11, 3.5))
    prof_c12["part_manquants"].plot.bar(ax=axes_12[0], color="tab:red", title="part de valeurs manquantes")
    prof_c12["part_mode"].plot.bar(ax=axes_12[1], color="tab:blue", title="part de la valeur la plus fréquente")
    for ax_12 in axes_12:
        ax_12.set_ylim(0, 1.05 if ax_12 is axes_12[1] else None)
    plt.tight_layout()
    plt.show()
wb.check("29.12a", most_missing_12)
wb.check("29.12b", duplicates_12)
wb.check("29.12c", majority_12)

### Ex 29.13 — Un score trop beau pour être vrai : doublons entre entraînement et test 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver les trois fuites d'une évaluation et la corriger, en découpant par groupes.  
**Prérequis :** ch. 8 (découpage, validation) · fiche « Au-delà du livre : auditer un dataset » · 💼 29.E3 · **Parcours :** R, M, C

Un collègue a photographié 500 vêtements, chacun quatre fois (lumière et bruit un peu différents) : 2 000 images, `X_13`, leurs labels `Y_13`, et `GROUPS_13`, le numéro du vêtement de chaque image. Son évaluation, `evaluate_buggy_13` (cellule suivante), annonce une accuracy de test parfaite. Elle contient **trois** lignes fautives, trois façons de laisser le test influencer l'entraînement ou l'évaluation.

1. Écris dans `faulty_13` les lettres des lignes fautives (dans n'importe quel ordre).
2. Mesure la fuite principale : `shared_13`, le nombre de lignes de test du découpage de la ligne B dont le vêtement (`GROUPS_13`) apparaît aussi dans l'entraînement. Refais ce découpage avec les mêmes réglages : `train_test_split` rend les mêmes indices pour toute liste de même longueur, découpée avec le même `stratify` et le même `random_state`.
3. Écris `evaluate_fixed_13` selon sa docstring : mêmes sorties, sans les fuites.

La cellule de vérification lance aussi ta version sur un piège : un label tiré **au hasard** pour chaque vêtement. Rien n'est apprenable, une évaluation honnête doit tomber au niveau du hasard (environ 0,1).

Dans tes notes : pour chaque ligne fautive, la fuite qu'elle crée. Laquelle explique presque tout l'écart ? Où trouve-t-on des quasi-doublons dans de vrais datasets (photos en rafale, plusieurs images d'un même patient, phrases traduites deux fois) ?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier


def make_near_duplicates_13(n_sources=500, copies=4, seed=13):
    """Photos taken several times: each of n_sources Fashion-MNIST images appears `copies` times, each time with a
    slightly different brightness and a small noise. X uint8 (N, 28, 28), y (N,), groups (N,): the index of the
    source image of each row (the rows of a group are near-duplicates)."""
    Xs, ys = wb.datasets.load_fashion_mnist("train", n_sources, seed=seed)
    rng = np.random.default_rng(seed)
    n = n_sources * copies
    X = np.repeat(Xs.astype(float), copies, axis=0) * rng.uniform(0.9, 1.1, (n, 1, 1)) + rng.normal(0, 6, (n, 28, 28))
    return np.clip(np.rint(X), 0, 255).astype(np.uint8), np.repeat(ys, copies), np.repeat(np.arange(n_sources), copies)


X_13, Y_13, GROUPS_13 = make_near_duplicates_13()


# The evaluation to debug (do not change this cell: copy the function into the next one to fix it)
def evaluate_buggy_13(X, y, groups):
    X = X.reshape(len(X), -1) / 255.0                                                                    # A
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)         # B
    pca = PCA(n_components=40, random_state=0).fit(X)                                                    # C
    X_tr, X_te = pca.transform(X_tr), pca.transform(X_te)                                                # D
    scores = {k: KNeighborsClassifier(k).fit(X_tr, y_tr).score(X_te, y_te) for k in (1, 3, 5, 9)}      # E
    best = max(scores, key=scores.get)                                                                   # F
    return best, scores[best]                                                                            # G


best_b13, score_b13 = evaluate_buggy_13(X_13, Y_13, GROUPS_13)
print(f"{len(X_13)} images, {len(np.unique(GROUPS_13))} sources ; évaluation à corriger : k = {best_b13}, accuracy de test {fr(score_b13)}")

In [ ]:
faulty_13 = ...   # a) the letters of the faulty lines of evaluate_buggy_13, e.g. "AB"
shared_13 = ...   # b) the number of test rows of the split of line B whose source image (GROUPS_13) is also in the training rows


def evaluate_fixed_13(X, y, groups):
    """evaluate_buggy_13 without its leaks (same returns: the chosen k, the test accuracy):
    - test split by groups: GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0);
    - k chosen on a validation part carved out of the training rows, also by groups (random_state=1);
    - the PCA fitted on the rows that train the model, never on the rows it is scored on;
    - at the end, k-NN with the chosen k refitted on all the training rows, scored once on the test rows."""
    raise NotImplementedError("evaluate_fixed_13")

In [ ]:
wb.check("29.13a", faulty_13)
wb.check("29.13b", shared_13)
with checked("29.13"):
    out_13 = evaluate_fixed_13(X_13, Y_13, GROUPS_13)
    if not returned("29.13", "evaluate_fixed_13", out_13):
        raise StopCheck
    best_13, score_13 = out_13
    verdict("29.13", best_13 in (1, 3, 5, 9) and 0 <= score_13 <= 1, f"k = {best_13}, accuracy de test {fr(score_13)}.",
            f"attendu (k parmi 1, 3, 5, 9 ; une accuracy entre 0 et 1), reçu {out_13}.")
    # A trap: one random label per source image. Nothing can be learned, so an honest evaluation gives about 0.1;
    # a split that leaves near-duplicates on both sides recognises the copies and scores near 1.
    y_rand13 = np.random.default_rng(1).integers(0, 10, GROUPS_13.max() + 1)[GROUPS_13]
    _, score_rand13 = evaluate_fixed_13(X_13, y_rand13, GROUPS_13)
    _, score_rand_b13 = evaluate_buggy_13(X_13, y_rand13, GROUPS_13)
    verdict("29.13", score_rand13 < 0.3,
            f"labels tirés au hasard par source : {fr(score_rand13)} (le hasard), contre {fr(score_rand_b13)} pour la version fautive.",
            f"labels tirés au hasard par source : ta version obtient {fr(score_rand13)}, loin du hasard (0,1) : des copies "
            "d'une même source sont encore des deux côtés du découpage.")
    verdict("29.13", 0.5 < score_13 < score_b13 - 0.1, "sur les vraies données, l'accuracy honnête est nettement plus basse.",
            f"accuracy {fr(score_13)} : attendu nettement moins que les {fr(score_b13)} de la version fautive, et plus de 0,5.")

### Ex 29.14 — Décalage de distribution : un modèle MNIST face aux chiffres de scikit-learn 🔬 ★★★ ⏱️ 35 min
**Objectif :** mesurer ce que récupère une préparation fidèle des images, puis quelques exemples étiquetés, face à un décalage de distribution.  
**Prérequis :** Ex 29.9 · ch. 23 (fine-tuning) · fiche « Au-delà du livre : le décalage de distribution » · **Parcours :** R, M, C

En 29.9, les chiffres de scikit-learn, agrandis tels quels, ont dérouté `MODEL_M`. Or les images de MNIST suivent une recette précise : chaque chiffre est réduit pour tenir dans un carré de 20 × 20, puis placé dans l'image de 28 × 28 de sorte que son **centre de masse** soit au centre.

**Contrat** de `to_mnist_14(images, method)` : la docstring de la cellule suivante. Trois méthodes : `"naive"` (comme `digits_naive`), `"box"` (agrandir à 20 × 20, puis 4 pixels noirs de chaque côté) et `"center"` (`"box"`, puis chaque image décalée pour mettre son centre de masse au centre, `(13,5 ; 13,5)`).

La cellule de vérification contrôle les trois méthodes. L'expérience mesure ensuite :
1. l'accuracy de `MODEL_M` avec chaque méthode ;
2. l'accuracy après un fine-tuning du réseau sur $k$ chiffres étiquetés par classe ($k$ = 0, 1, 5, 20), mesurée sur les autres chiffres ;
3. une référence : un modèle simple entraîné sur les chiffres de scikit-learn eux-mêmes.

Dans tes notes : quelle étape de préparation rapporte le plus ? Combien d'exemples étiquetés faut-il pour approcher la référence ? Le décalage restant tient-il aux images (le décalage de covariables de la fiche) ou aux labels ? Que ferais-tu, dans ton projet final, si les données d'utilisation diffèrent de celles d'entraînement ?

In [ ]:
from scipy import ndimage

In [ ]:
def to_mnist_14(images, method="box"):
    """Digits of scikit-learn (N, 8, 8), values 0 to 16, converted to the MNIST format: uint8 (N, 28, 28), 0 to 255
    (values * 255 / 16, rounded with np.rint, clipped to [0, 255] at the very end):
    - "naive": the whole 8 x 8 image enlarged to 28 x 28 (F.interpolate, mode="bilinear", align_corners=False);
    - "box": enlarged to 20 x 20 the same way, then 4 black pixels added on each side (MNIST digits fit in 20 x 20);
    - "center": "box", then each image shifted (ndimage.shift, order=1, float values) so that its centre of mass
      (ndimage.center_of_mass) lands on the centre (13.5, 13.5)."""
    raise NotImplementedError("to_mnist_14")

In [ ]:
with checked("29.14"):
    few_14 = DIGITS_9.images[:20]
    naive_14 = to_mnist_14(few_14, "naive")
    if not returned("29.14", "to_mnist_14", naive_14):
        raise StopCheck
    verdict("29.14", isinstance(naive_14, np.ndarray) and naive_14.shape == (20, 28, 28) and naive_14.dtype == np.uint8,
            "uint8 de forme (N, 28, 28).", f"attendu uint8 (20, 28, 28), reçu {getattr(naive_14, 'dtype', type(naive_14))} {getattr(naive_14, 'shape', '')}.")
    same("29.14", "naive (comparé à digits_naive des outils)", naive_14.astype(float), digits_naive(few_14).astype(float), atol=1.0)
    box_14 = to_mnist_14(few_14, "box")
    border_14 = np.concatenate([box_14[:, :4].ravel(), box_14[:, -4:].ravel(), box_14[:, :, :4].ravel(), box_14[:, :, -4:].ravel()])
    verdict("29.14", border_14.max() == 0 and box_14[:, 4:24, 4:24].max() > 200,
            "box : un cadre noir de 4 pixels autour d'un chiffre de 20 × 20.",
            "box : attendu 4 pixels noirs de chaque côté et le chiffre agrandi au centre, en 20 × 20.")
    t_14 = torch.as_tensor(few_14, dtype=torch.float32)[:, None] * 255.0 / 16.0
    want_box_14 = np.clip(np.rint(F.interpolate(t_14, size=(20, 20), mode="bilinear", align_corners=False)[:, 0].numpy()), 0, 255)
    same("29.14", "box (le carré central de 20 × 20)", box_14[:, 4:24, 4:24].astype(float), want_box_14, atol=1.0)
    center_14 = to_mnist_14(few_14, "center")
    mass_14 = np.array([ndimage.center_of_mass(img.astype(float)) for img in center_14])
    verdict("29.14", np.abs(mass_14 - 13.5).max() < 0.5, "center : le centre de masse de chaque chiffre est au centre de l'image.",
            f"center : un centre de masse s'écarte de {fr(float(np.abs(mass_14 - 13.5).max()), 2)} pixel du centre (13,5 ; 13,5).")

In [ ]:
with checked("29.14"):
    # 1. The three conversions, the same network trained on MNIST (MODEL_M, 29.8): what does each step bring?
    methods_14 = ["naive", "box", "center"]
    converted_14 = {m: to_mnist_14(DIGITS_9.images, m) for m in methods_14}
    show_digits([XM_TEST[:8]] + [converted_14[m][:8] for m in methods_14], ["MNIST"] + methods_14)
    for m in methods_14:
        acc_14, conf_14 = evaluate(MODEL_M, converted_14[m], YD_9)
        print(f"{m:7s} : accuracy {fr(acc_14)}, confiance moyenne {fr(conf_14)}")

    # 2. Adapt with a few labelled digits of the new source (k per class), scored on the other digits
    rng_14 = np.random.default_rng(0)
    X_best14 = converted_14["center"]
    for k_14 in (0, 1, 5, 20):
        idx_14 = np.concatenate([rng_14.permutation(np.flatnonzero(YD_9 == c))[:k_14] for c in range(10)]).astype(int)
        rest_14 = np.setdiff1d(np.arange(len(YD_9)), idx_14)
        model_14 = fine_tune(MODEL_M, X_best14[idx_14], YD_9[idx_14]) if k_14 else MODEL_M
        print(f"{k_14:2d} chiffres étiquetés par classe : accuracy {fr(evaluate(model_14, X_best14[rest_14], YD_9[rest_14])[0])}")

    # 3. The reference: a model trained on the new source itself (logistic regression on the 8 x 8 pixels)
    from sklearn.linear_model import LogisticRegression
    Xa_14, Xb_14, ya_14, yb_14 = train_test_split(DIGITS_9.data, YD_9, test_size=0.25, random_state=0, stratify=YD_9)
    print(f"référence entraînée sur les chiffres de scikit-learn : accuracy {fr(LogisticRegression(max_iter=3000).fit(Xa_14, ya_14).score(Xb_14, yb_14))}")

### Ex 29.15 — Data card et script de téléchargement vérifié pour le projet final 🛠️ ★★★ ⏱️ 60 min
**Objectif :** écrire un téléchargement reproductible, vérifié par somme de contrôle, et la data card du dataset de ton projet final.  
**Prérequis :** Ex 29.6, 29.11 · fiche « Au-delà du livre : documenter et télécharger » (encadré 🧮) · **Parcours :** R, M, C

**Contrat**
- `sha256_15(path, chunk_size=1 << 20)` : la somme de contrôle SHA-256 d'un fichier (64 caractères hexadécimaux), lu **par morceaux** (`hashlib.sha256`, `update`, `hexdigest`) : un fichier de plusieurs gigaoctets ne tient pas en mémoire ;
- `download_verified_15(url, dest, sha256)` : si `dest` existe déjà avec la bonne somme, ne rien télécharger ; sinon télécharger dans `dest + ".part"` (`urllib.request.urlopen`), vérifier la somme, puis renommer en `dest` ; si la somme est fausse, supprimer le `.part` et lever `ValueError` (`dest` reste tel quel). Rendre `Path(dest)`.

La cellule de vérification « télécharge » un fichier local (une URL `file://`) : arrivée intacte, pas de nouveau téléchargement si le fichier est déjà là, refus d'une somme fausse.

Puis remplis `CARD_15`, la data card du dataset de **ton projet final** : les six sections (Provenance, Contenu, Licence, Usage prévu, Limites et biais, Téléchargement), l'adresse de la source, et la somme SHA-256 de ton fichier (calculée avec `sha256_15` une fois le fichier téléchargé). Pour un dataset du Hugging Face Hub, note aussi la révision (`HfApi().dataset_info(nom).sha`) et télécharge avec `hf_hub_download(..., repo_type="dataset", revision=...)`. Modèle : les fiches de `data/cards/`. La vérification liste ce qui manque.

Dans tes notes : pourquoi télécharger dans un `.part` puis renommer ? Une somme de contrôle protège-t-elle contre un dataset qui change de contenu, contre un fichier corrompu, contre un dataset retiré ? Que doit faire ton script si la source disparaît ?

In [ ]:
def sha256_15(path, chunk_size=1 << 20):
    """The SHA-256 of a file, 64 hexadecimal characters, read by chunks of chunk_size bytes (never all at once)."""
    raise NotImplementedError("sha256_15")


def download_verified_15(url, dest, sha256):
    """Download `url` to `dest`, unless dest already exists with the right SHA-256 (then nothing is downloaded).
    Download into dest + ".part" (urllib.request.urlopen), check its SHA-256, then rename it to dest; on a wrong
    SHA-256, delete the .part file and raise ValueError (dest is left as it was). Return Path(dest)."""
    raise NotImplementedError("download_verified_15")


# The data card of YOUR final project dataset: replace every "..." (the check lists what is missing)
CARD_15 = """# ...

## Provenance
...

## Contenu
...

## Licence
...

## Usage prévu
...

## Limites et biais
...

## Téléchargement
...
"""

In [ ]:
with checked("29.15"):
    src_15 = WORK_DIR / "source_15.bin"
    src_15.write_bytes(np.random.default_rng(15).bytes(3_000_000))           # a 3 MB file to "download"
    good_15 = hashlib.sha256(src_15.read_bytes()).hexdigest()
    got_15 = sha256_15(src_15)
    if not returned("29.15", "sha256_15", got_15):
        raise StopCheck
    verdict("29.15", got_15 == good_15, "sha256_15 rend la bonne somme de contrôle.",
            f"sha256_15 : attendu {good_15[:16]}…, reçu {str(got_15)[:16]}… (hexdigest(), sur tout le fichier).")
    dest_15 = WORK_DIR / "copy_15.bin"
    path_15 = download_verified_15(src_15.as_uri(), dest_15, good_15)
    verdict("29.15", Path(path_15) == dest_15 and dest_15.exists() and sha256_15(dest_15) == good_15,
            "un téléchargement vérifié arrive à destination.", "le fichier n'est pas arrivé intact dans dest, ou le chemin rendu est faux.")
    verdict("29.15", not dest_15.with_name(dest_15.name + ".part").exists(), "aucun fichier .part ne reste.",
            "un fichier .part reste après un téléchargement réussi : renomme-le en dest.")
    src_15.write_bytes(b"un autre contenu")                                   # the source changes...
    try:
        download_verified_15(src_15.as_uri(), dest_15, good_15)              # ...but dest is already there, intact
        skipped_15 = sha256_15(dest_15) == good_15
    except ValueError:
        skipped_15 = False
    verdict("29.15", skipped_15, "un fichier déjà présent et intact n'est pas téléchargé de nouveau.",
            "dest était déjà là avec la bonne somme : il ne fallait rien télécharger.")
    bad_15 = WORK_DIR / "bad_15.bin"
    try:
        download_verified_15(src_15.as_uri(), bad_15, good_15)
        raised_15 = False
    except ValueError:
        raised_15 = True
    verdict("29.15", raised_15 and not bad_15.exists() and not bad_15.with_name(bad_15.name + ".part").exists(),
            "une somme fausse lève ValueError et ne laisse aucun fichier.",
            "une somme fausse doit lever ValueError, sans laisser dest ni le .part.")
    problems_15 = check_card(CARD_15)
    if "..." in CARD_15:
        print("⏳ Ex 29.15 : la data card de ton projet n'est pas encore remplie (il reste des « ... »).")
    else:
        verdict("29.15", not problems_15, "la data card a ses six sections, une source et une somme de contrôle.",
                "data card incomplète : " + " ; ".join(problems_15) + ".")

### Ex 29.16 — Le dataset piégé : trouver les huit défauts 🏆 ★★★ ⏱️ 60 min
**Objectif :** auditer seul un tableau inconnu, y trouver huit défauts de types différents, puis le nettoyer.  
**Prérequis :** Ex 29.12, 29.13 · fiche « Au-delà du livre : auditer un dataset » (tableau des défauts) · **Parcours :** C

`FRUITS_16` décrit 1 230 fruits : un identifiant, six mesures, un pays, un code qualité, le label `fruit` (pomme, poire ou coing) et le découpage `split` déjà fait (`"train"` ou `"test"`). Il cache **huit défauts**, un de chaque sorte : doublons exacts, quasi-doublons entre entraînement et test, valeurs manquantes codées par un nombre, colonne sans information, colonne qui trahit le label, unités incohérentes, labels mal écrits, classe très rare.

Explore-le dans autant de cellules que tu veux (`describe`, `value_counts`, `duplicated`, `nunique`, `crosstab`…), puis réponds aux questions a) à h) de la cellule à compléter et écris `clean_16(df)`, qui rend une copie nettoyée.

La cellule de vérification contrôle tes huit réponses, puis ta copie nettoyée. L'expérience entraîne la même forêt aléatoire sur les données brutes et sur les données nettoyées, et compare leurs accuracies de test.

Dans tes notes : pour chaque défaut, la commande qui l'a révélé et ce qu'il aurait fait au modèle. Pourquoi l'accuracy baisse-t-elle après le nettoyage, alors que le modèle est meilleur ? Que mettrais-tu dans la section « Limites et biais » de la data card de ce dataset ?

In [ ]:
FRUITS_16 = wb.datasets.make_trap_dataset()   # 1 230 rows, 11 columns; data/cards/trap.md
FRUITS_16.head()

In [ ]:
# Explore FRUITS_16 in as many cells as you like (describe, value_counts, duplicated, groupby...), then answer:
exact_16 = ...     # a) the number of rows that repeat an earlier row exactly
near_16 = ...      # b) the number of "test" rows that have a near-duplicate among the "train" rows: identical in every
                   #    column except id_lot, split and poids_g, and poids_g within 1 g
missing_16 = ...   # c) the column whose missing values are coded by a number (its name)
constant_16 = ...  # d) the column that carries no information at all
leak_16 = ...      # e) the column that gives the label away
units_16 = ...     # f) the column where some values are in another unit
labels_16 = ...    # g) the number of rows whose label is not one of "pomme", "poire", "coing"
rare_16 = ...      # h) the class (correctly written) with the fewest rows


def clean_16(df):
    """A cleaned copy of df: exact duplicates and the test near-duplicates of b) removed, missing values back to NaN,
    the columns of d) and e) dropped, the values of f) in a single unit, the labels of g) corrected (or the rows
    removed). df itself is not modified."""
    raise NotImplementedError("clean_16")

In [ ]:
wb.check("29.16a", exact_16)
wb.check("29.16b", near_16)
wb.check("29.16c", missing_16)
wb.check("29.16d", constant_16)
wb.check("29.16e", leak_16)
wb.check("29.16f", units_16)
wb.check("29.16g", labels_16)
wb.check("29.16h", rare_16)
with checked("29.16"):
    before_16 = FRUITS_16.copy()
    clean_c16 = clean_16(FRUITS_16)
    if not returned("29.16", "clean_16", clean_c16):
        raise StopCheck
    verdict("29.16", FRUITS_16.equals(before_16), "le tableau d'origine n'a pas été modifié.",
            "clean_16 a modifié FRUITS_16 : travaille sur une copie (df.copy()).")
    answers_16 = wb.datasets.trap_dataset_answers()
    gone_16 = [c for c in (answers_16["constant_column"], answers_16["leaking_column"]) if c in clean_c16.columns]
    verdict("29.16", not gone_16, "les colonnes inutile et fuyante ont disparu.", f"colonnes encore présentes : {gone_16}.")
    cols_c16 = [c for c in clean_c16.columns if c != "id_lot"]
    verdict("29.16", not clean_c16[cols_c16].duplicated().any(), "plus aucun doublon exact.",
            f"{int(clean_c16[cols_c16].duplicated().sum())} ligne(s) encore en double.")
    verdict("29.16", not (clean_c16["acidite"] == -999).any(), "plus de −999 dans acidite.",
            "il reste des −999 dans acidite : remplace-les par NaN (ou retire ces lignes).")
    verdict("29.16", clean_c16["diametre_cm"].max() < 30, "les diamètres sont tous en centimètres.",
            f"un diamètre vaut encore {fr(float(clean_c16['diametre_cm'].max()), 1)} : convertis les millimètres.")
    verdict("29.16", set(clean_c16["fruit"]) <= {"pomme", "poire", "coing"}, "tous les labels sont valides.",
            f"labels encore invalides : {sorted(set(clean_c16['fruit']) - {'pomme', 'poire', 'coing'})}.")
    tr_c16, te_c16 = clean_c16[clean_c16["split"] == "train"], clean_c16[clean_c16["split"] == "test"]
    key_c16 = [c for c in clean_c16.columns if c not in ("id_lot", "split", "poids_g")]
    left_c16 = te_c16.merge(tr_c16[key_c16], on=key_c16)
    verdict("29.16", len(left_c16) == 0, "aucun quasi-doublon de l'entraînement ne reste dans le test.",
            f"{len(left_c16)} ligne(s) de test ont encore leur quasi-doublon dans l'entraînement.")
    verdict("29.16", len(clean_c16) >= 1150, f"{len(clean_c16)} lignes gardées sur {len(FRUITS_16)}.",
            f"seulement {len(clean_c16)} lignes gardées : as-tu jeté plus que les défauts ?")

In [ ]:
# What the defects change: the same forest, trained on "train" and scored on "test", before and after cleaning
from sklearn.ensemble import RandomForestClassifier


def score_16(df, columns):
    """Accuracy on the "test" rows of a random forest trained on the "train" rows (missing values set to -1)."""
    data = df.copy()
    X = pd.get_dummies(data[columns], dtype=float).fillna(-1)
    train, test = data["split"] == "train", data["split"] == "test"
    forest = RandomForestClassifier(n_estimators=100, random_state=0).fit(X[train], data.loc[train, "fruit"])
    return forest.score(X[test], data.loc[test, "fruit"])


with checked("29.16"):
    dirty_cols_16 = [c for c in FRUITS_16.columns if c not in ("id_lot", "split", "fruit")]
    clean_e16 = clean_16(FRUITS_16)
    clean_cols_16 = [c for c in clean_e16.columns if c not in ("id_lot", "split", "fruit")]
    print(f"données brutes, toutes les colonnes : accuracy {fr(score_16(FRUITS_16, dirty_cols_16))}")
    print(f"données nettoyées                   : accuracy {fr(score_16(clean_e16, clean_cols_16))}")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu charger un même dataset par deux API et vérifier qu'elles rendent les mêmes données ?
2. Sais-tu dire, en lisant sa page, si un dataset peut servir à ton projet (licence, provenance, retraits) ?
3. Sais-tu repérer une fuite entre entraînement et test, et découper par groupes ?
4. As-tu choisi le dataset de ton projet final, avec sa data card et un téléchargement vérifié ?

**Pour aller plus loin** : Gebru et coll., « Datasheets for Datasets », et l'article de Northcutt et coll. sur les erreurs de labels des benchmarks, cités dans la fiche. La suite : le checkpoint de la partie VI, puis le projet final.